# Final Analysis
Insert this code into the final report – I will try to comment everything and make our steps clear!

## 1. Preprocess Data

In [109]:
import pandas as pd
import geopandas as gpd
import dask_geopandas as dgpd
import matplotlib.pyplot as plt
import osmnx as ox
import networkx as nx
import shapely
import contextily as ctx

In [110]:
# create bounding box and constant variables
USA_48_STATES = {
    "min_long": -128.320313,
    "max_long": -65.039063,
    "min_lat": 24.367114,
    "max_lat": 50.401515
}

USA_48_STATES_BBOX = (USA_48_STATES["min_long"], USA_48_STATES["min_lat"], USA_48_STATES["max_long"], USA_48_STATES["max_lat"])

ONE_MILE_IN_METERS = 1609.344

In [111]:
us_charger_locations = pd.read_csv('data/USChargingLocations.csv')
us_charger_locations = gpd.GeoDataFrame(us_charger_locations, 
                                        geometry = gpd.points_from_xy(
                                            us_charger_locations['Longitude'], 
                                            us_charger_locations['Latitude']))

# filters charging locations to 48 states
us_charger_locations = us_charger_locations.cx[USA_48_STATES["min_long"] : USA_48_STATES["max_long"], 
                                               USA_48_STATES["min_lat"] : USA_48_STATES["max_lat"]]
us_charger_locations = us_charger_locations.set_crs('EPSG:4326')

/var/folders/rd/8mbxk5gd789c6ptzwnn0v70w0000gn/T/ipykernel_63265/830304.py:1: DtypeWarning: Columns (20,40,64,69) have mixed types. Specify dtype option on import or set low_memory=False.
  us_charger_locations = pd.read_csv('data/USChargingLocations.csv')


In [112]:
interstates = gpd.read_file('data/spatial_interstates_w_mobility.geojson', bbox = USA_48_STATES_BBOX)
# this dataframe has almost 100 columns, so we are narrowing just to the important columns
interstates = interstates[[':id', 'aadt', 'sectionlength', 'geometry']]

# convert both to correct types
interstates.loc[:, 'sectionlength'] = interstates['sectionlength'].astype('float')
interstates.loc[:, 'aadt'] = interstates['aadt'].astype('float')

# makes calculations faster to simplify geometry/lines for processing. doesnt affect viewing from a national level
interstates = interstates.to_crs('EPSG:9311')
interstates.loc[:, 'geometry'] = interstates.simplify(50)
interstates = interstates[interstates.is_valid & ~interstates.is_empty]

/opt/anaconda3/lib/python3.12/site-packages/shapely/constructive.py:1177: RuntimeWarning: invalid value encountered in simplify_preserve_topology
  return lib.simplify_preserve_topology(geometry, tolerance, **kwargs)


## 2. Calculate Vehicle Miles Travelled for each Highway Segment

We are calculating Vehicle Miles Travelled (VMT) instead of just using AADT for mobility because this normalizes the data. To calculate VMT, we must have valid values for all AADT.

In [113]:
# seperate segments into two dataframes: those with a value for aadt, and those without a value for aadt
has_aadt = interstates[~interstates['aadt'].isna()].copy()
no_aadt = interstates[interstates['aadt'].isna()].copy()

In [114]:
# pull aadt value from the closest segement with an aadt value
resolved_aadt = gpd.sjoin_nearest(
    no_aadt.drop(columns='aadt'),
    has_aadt[['aadt', 'geometry']],
    how = 'left',
    max_distance = ONE_MILE_IN_METERS * 5
)

In [115]:
# apply these values back to the current dataframe
interstates.loc[resolved_aadt.index, 'aadt'] = resolved_aadt['aadt']

In [116]:
# calculate vmt
interstates['vmt'] = interstates['aadt'] * interstates['sectionlength']

In [117]:
# re-enforce types
interstates[['aadt', 'sectionlength', 'vmt']] = interstates[['aadt', 'sectionlength', 'vmt']].astype(float)

## 3. Find closest charger

In the context of road tripping, we want to make sure chargers are near the interstate. Since EVs can sometimes take hours to charge, we don't want to add another hour of driving just to get to a charger!

In [118]:
# create 5 mile buffers
interstates_5_miles = interstates.to_crs('EPSG:9311').buffer(ONE_MILE_IN_METERS * 5)

# filters chargers to only include ones 5 miles from the interstate
charging_locations_by_interstate = gpd.sjoin(
    us_charger_locations.to_crs('EPSG:9311'), 
    gpd.GeoDataFrame(geometry=interstates_5_miles), 
    how="inner", 
    predicate="intersects"
).drop_duplicates(subset='ID') 

/opt/anaconda3/lib/python3.12/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(


In [119]:
# create function to find distance to the nearest charger
def find_dist_to_nearest_charger(partition, points):
    if partition.empty:
        return partition
    
    return gpd.sjoin_nearest(
        partition, 
        points, 
        how="left", 
        distance_col="dist_to_charger"
    )

# uses dask to make this process much faster
# number of ideal partitions depends on number of cpu cores, and may need tweaking based on where you're running this
d_interstates = dgpd.from_geopandas(interstates, npartitions=24)

joined_delayed = d_interstates.map_partitions(
    find_dist_to_nearest_charger, 
    points=charging_locations_by_interstate[['geometry']]
)

# compute thing to find nearest charger
interstates_with_info = joined_delayed.compute(scheduler='threads')

# clean
interstates_with_info = interstates_with_info.drop_duplicates(subset=[':id'])
interstates_with_info['dist_charger_miles'] = interstates_with_info['dist_to_charger'] / ONE_MILE_IN_METERS

## Create overall score to find high priority EV deserts

In [120]:
interstates_with_info['score'] = interstates_with_info['vmt'] * interstates_with_info['dist_charger_miles']

In [121]:
# keep relevant columns
interstates_with_info = interstates_with_info[[':id', 'aadt', 'sectionlength', 'vmt', 'dist_charger_miles', 'score', 'geometry']]

In [122]:
interstates_with_info.to_file('data/interstates_with_charger_info.geojson', driver='GeoJSON')

## Find recommendations for areas that need EV chargers

In [123]:
# load zip code shapefile
zcta = gpd.read_file('data/zcta_shapefile/tl_2025_us_zcta520.shp')
# interstates_with_info = gpd.read_file('data/interstates_with_charger_info.geojson')

In [124]:
# only keep relevant cols
zcta = zcta[['ZCTA5CE20', 'geometry']]
zcta = zcta.rename({"ZCTA5CE20": "zip_code"}, axis = 1)

# re-project to equal-area and meters, and fix typing
zcta = zcta.to_crs('EPSG:9311')
zcta.loc[:, 'zip_code'] = zcta['zip_code'].astype('int')
# interstates_with_info = interstates_with_info.to_crs('EPSG:9311')

To make this easier for segements between zip codes, we will use the centroid of each segment to find which zip code it belongs to.

In [125]:
interstates_with_info['centroid'] = interstates_with_info.geometry.centroid

# spatial join zip-code to each segement
interstates_with_info = gpd.sjoin(
    interstates_with_info.set_geometry('centroid'),
    zcta,
    how = "left",
    predicate = "within"
)

In [ ]:
# for viz - feel free to delete for report
interstates_with_info = interstates_with_info.drop('centroid', axis = 1)
interstates_with_info.to_file('data/interstates_with_charger_info.geojson', driver='GeoJSON')

In [126]:
# find highest priority zipcodes
high_priority_zipcodes = interstates_with_info.groupby('zip_code')['score'].mean()
top_10_deserts_zipcodes = high_priority_zipcodes.sort_values(ascending=False)[0:10].index.to_list()
top_10_deserts_zipcodes

[41098, 99356, 87120, 36613, 56389, 29377, 87943, 59435, 7974, 36109]